In [2]:
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)  # seed berbeda tiap kota agar datanya bervariasi, namun tetap konsisten/reproducible
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris

Ketiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.


In [3]:
!hdfs dfs -mkdir -p /user/nayla/ecommerce/raw
!hdfs dfs -mkdir -p /user/nayla/ecommerce/processed
!hdfs dfs -ls -R /user/nayla/ecommerce

drwxr-xr-x   - nayla supergroup          0 2026-09-10 10:12 /user/nayla/ecommerce/processed
drwxr-xr-x   - nayla supergroup          0 2026-09-10 10:12 /user/nayla/ecommerce/raw


In [4]:
!hdfs dfs -put transaksi_magelang.csv transaksi_yogyakarta.csv transaksi_semarang.csv /user/nayla/ecommerce/raw/
!hdfs dfs -ls -h /user/nayla/ecommerce/raw

Found 3 items
-rw-r--r--   1 nayla supergroup     12.0 K 2026-09-10 10:14 /user/nayla/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 nayla supergroup     11.9 K 2026-09-10 10:14 /user/nayla/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 nayla supergroup     12.4 K 2026-09-10 10:14 /user/nayla/ecommerce/raw/transaksi_yogyakarta.csv


In [6]:
import subprocess
import io
import pandas as pd

def baca_csv_dari_hdfs(path_hdfs):
    hasil = subprocess.run(["hdfs", "dfs", "-cat", path_hdfs], stdout=subprocess.PIPE)
    return pd.read_csv(io.BytesIO(hasil.stdout))

df_magelang = baca_csv_dari_hdfs("/user/nayla/ecommerce/raw/transaksi_magelang.csv")
df_yogyakarta = baca_csv_dari_hdfs("/user/nayla/ecommerce/raw/transaksi_yogyakarta.csv")
df_semarang = baca_csv_dari_hdfs("/user/nayla/ecommerce/raw/transaksi_semarang.csv")

df_gabungan = pd.concat([df_magelang, df_yogyakarta, df_semarang], ignore_index=True)
print(df_gabungan["kota"].value_counts())

kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64


In [9]:
df_gabungan["total_pendapatan"] = df_gabungan["unit_terjual"] * df_gabungan["harga_satuan"]

ringkasan_kota_kategori = (
    df_gabungan.groupby(["kota", "kategori"])["total_pendapatan"]
    .sum()
    .reset_index()
    .sort_values(["kota", "total_pendapatan"], ascending=[True, False])
)

df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
ringkasan_kota_kategori.to_csv("ringkasan_kota_kategori.csv", index=False)
!hdfs dfs -put data_gabungan_bersih.csv ringkasan_kota_kategori.csv /user/nayla/ecommerce/processed/
!hdfs dfs -ls -h /user/nayla/ecommerce/processed

Found 2 items
-rw-r--r--   1 nayla supergroup     40.3 K 2026-09-10 10:22 /user/nayla/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 nayla supergroup        530 2026-09-10 10:22 /user/nayla/ecommerce/processed/ringkasan_kota_kategori.csv
